In [1]:
import os
os.chdir("..")
import numpy as np
import pandas as pd
from scipy import stats

In [2]:
scores_df = pd.read_csv("outputs/per_fold_f1_scores.csv")
summary = pd.read_csv("outputs/model_comparison_results.csv")

best_model = summary.iloc[0]["Model"]
if best_model == "RuleEngine":
    best_model = summary.iloc[1]["Model"]

print("Best model overall:", best_model)
print("n per model:", len(scores_df))
print("\nFull summary:")
print(summary.to_string(index=False))

Best model overall: Proposed_Stacking_Ensemble
n per model: 50

Full summary:
                     Model  Mean_Macro_F1      Std
Proposed_Stacking_Ensemble       0.749279 0.111944
                   XGBoost       0.748420 0.115831
        LogisticRegression       0.743812 0.130790
              RandomForest       0.730598 0.091181
          GradientBoosting       0.723334 0.113563
                       MLP       0.683362 0.109444
                RuleEngine       0.667595 0.129009
                       SVM       0.652526 0.119232


In [3]:
def wilcoxon_test(a, b, name_a, name_b):
    stat, p = stats.wilcoxon(a, b)
    diffs = np.array(a) - np.array(b)
    ranks = stats.rankdata(np.abs(diffs)) * np.sign(diffs)
    r_rb = np.sum(ranks) / (len(diffs) * (len(diffs) + 1) / 2)
    print(f"{name_a} (mean={np.mean(a):.3f}) vs {name_b} (mean={np.mean(b):.3f})")
    print(f"  p={p:.4f}, effect size r={r_rb:.3f}")
    return {"comparison": f"{name_a}_vs_{name_b}", "p_value": p, "effect_size": r_rb}

# Headline comparison — this is the core claim of your paper
result = wilcoxon_test(scores_df[best_model], scores_df["RuleEngine"], best_model, "RuleEngine")

Proposed_Stacking_Ensemble (mean=0.749) vs RuleEngine (mean=0.668)
  p=0.0000, effect size r=0.810


In [4]:
results = [result]
model_cols = [c for c in scores_df.columns if c != "RuleEngine"]
for i in range(len(model_cols)):
    for j in range(i + 1, len(model_cols)):
        results.append(wilcoxon_test(
            scores_df[model_cols[i]], scores_df[model_cols[j]],
            model_cols[i], model_cols[j]
        ))

results_df = pd.DataFrame(results)
results_df["significant"] = results_df["p_value"] < 0.05
results_df.to_csv("outputs/statistical_test_results.csv", index=False)

print(f"\nSaved {len(results_df)} comparisons to statistical_test_results.csv")
print(f"\nSignificant comparisons (p<0.05): {results_df['significant'].sum()} out of {len(results_df)}")
print(results_df[results_df["significant"]][["comparison", "p_value", "effect_size"]].to_string(index=False))

LogisticRegression (mean=0.744) vs SVM (mean=0.653)
  p=0.0000, effect size r=0.752
LogisticRegression (mean=0.744) vs RandomForest (mean=0.731)
  p=0.4370, effect size r=0.126
LogisticRegression (mean=0.744) vs GradientBoosting (mean=0.723)
  p=0.1462, effect size r=0.236
LogisticRegression (mean=0.744) vs MLP (mean=0.683)
  p=0.0055, effect size r=0.451
LogisticRegression (mean=0.744) vs XGBoost (mean=0.748)
  p=0.4900, effect size r=-0.112
LogisticRegression (mean=0.744) vs Proposed_Stacking_Ensemble (mean=0.749)
  p=0.5606, effect size r=-0.100
SVM (mean=0.653) vs RandomForest (mean=0.731)
  p=0.0000, effect size r=-0.826
SVM (mean=0.653) vs GradientBoosting (mean=0.723)
  p=0.0000, effect size r=-0.704
SVM (mean=0.653) vs MLP (mean=0.683)
  p=0.0000, effect size r=-0.760
SVM (mean=0.653) vs XGBoost (mean=0.748)
  p=0.0000, effect size r=-0.879
SVM (mean=0.653) vs Proposed_Stacking_Ensemble (mean=0.749)
  p=0.0000, effect size r=-0.918
RandomForest (mean=0.731) vs GradientBoosting 